In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengenalan kepada primitif

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Keluaran beta model pelaksanaan baharu kini tersedia. Model pelaksanaan terarah memberikan lebih fleksibiliti semasa menyesuaikan aliran kerja mitigasi ralat anda. Lihat panduan [Model pelaksanaan terarah](/guides/directed-execution-model) untuk maklumat lanjut.

<span id="qpu-access-patterns"></span>
## Mengapa Qiskit memperkenalkan primitif?
Sama seperti zaman awal komputer klasikal, apabila pembangun terpaksa memanipulasi daftar CPU secara terus, antara muka awal kepada QPU hanya mengembalikan data mentah dari elektronik kawalan.
Ini tidak menjadi isu besar apabila QPU berada dalam makmal dan hanya membenarkan akses langsung oleh penyelidik.
Menyedari bahawa kebanyakan pembangun tidak akan dan tidak seharusnya biasa dengan menyuling data mentah tersebut kepada 0 dan 1, Qiskit memperkenalkan `backend.run`, abstraksi pertama untuk mengakses QPU di awan. Ini membolehkan pembangun beroperasi pada format data yang biasa dan menumpukan pada gambaran yang lebih besar.

Apabila akses kepada QPU menjadi lebih meluas, dan dengan lebih banyak algoritma kuantum dibangunkan, keperluan untuk abstraksi peringkat lebih tinggi timbul semula. Sebagai respons, Qiskit memperkenalkan antara muka primitif, yang dioptimumkan untuk dua tugas teras dalam pembangunan algoritma kuantum: anggaran nilai jangkaan (`Estimator`) dan pensampelan Circuit (`Sampler`). Matlamatnya sekali lagi adalah untuk membantu pembangun lebih menumpukan pada inovasi dan kurang pada penukaran data. Antara muka primitif menggantikan antara muka `backend.run`, kerana `Sampler` menyediakan akses perkakasan langsung yang sama seperti yang ditawarkan oleh `backend.run`.

## Apakah primitif?
Sistem pengkomputeran dibina di atas pelbagai lapisan abstraksi. Abstraksi membolehkan anda menumpukan pada tahap perincian tertentu yang relevan dengan tugas yang sedang dijalankan. Semakin hampir anda ke perkakasan, semakin rendah tahap abstraksi yang anda perlukan (contohnya, anda mungkin perlu memindahkan atau memanipulasi data pada peringkat arahan CPU). Semakin kompleks tugas yang ingin anda lakukan, semakin tinggi tahap abstraksinya (contohnya, anda mungkin menggunakan pustaka pengaturcaraan untuk melakukan pengiraan algebra).

Dalam konteks ini, *primitif* ialah arahan pemprosesan terkecil, blok binaan paling mudah yang boleh digunakan untuk mencipta sesuatu yang berguna untuk tahap abstraksi tertentu.

Kemajuan terkini dalam pengkomputeran kuantum telah meningkatkan keperluan untuk bekerja pada tahap abstraksi yang lebih tinggi. Apabila bidang ini bergerak ke arah unit pemprosesan kuantum (QPU) yang lebih besar dan aliran kerja yang lebih kompleks, tumpuan beralih daripada berinteraksi dengan isyarat Qubit individu kepada melihat peranti kuantum sebagai sistem yang menjalankan tugas yang diperlukan.

Dua tugas yang paling biasa untuk komputer kuantum adalah pensampelan keadaan kuantum dan pengiraan nilai jangkaan. Tugas-tugas ini mendorong reka bentuk primitif Qiskit: **Estimator** dan **Sampler**.

- Estimator mengira nilai jangkaan observable berkenaan keadaan yang disediakan oleh Circuit kuantum.
- Sampler mengambil sampel daftar output daripada pelaksanaan Circuit kuantum.

Ringkasnya, model pengkomputeran yang diperkenalkan oleh primitif Qiskit membawa pengaturcaraan kuantum selangkah lebih hampir ke tahap pengaturcaraan klasikal hari ini, di mana tumpuan kurang pada perincian perkakasan dan lebih pada keputusan yang ingin dicapai.

## Definisi dan pelaksanaan primitif
Terdapat dua jenis primitif Qiskit: kelas asas, dan pelaksanaannya. Primitif Qiskit ditakrifkan oleh kelas asas primitif sumber terbuka yang terdapat dalam Qiskit SDK (dalam modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Pembekal (seperti Qiskit Runtime) boleh menggunakan kelas asas ini untuk memperoleh pelaksanaan Sampler dan Estimator mereka sendiri. Kebanyakan pengguna akan berinteraksi dengan pelaksanaan pembekal, bukan primitif asas.

### Kelas asas
[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) dan [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Kelas asas abstrak yang menakrifkan antara muka biasa untuk melaksanakan primitif. Semua kelas lain dalam modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) mewarisi daripada kelas asas ini. Pembangun harus menggunakan ini jika mereka berminat untuk mencipta model pelaksanaan berasaskan primitif mereka sendiri untuk pembekal tertentu. Kelas-kelas ini mungkin juga berguna bagi mereka yang mahu melakukan pemprosesan yang sangat disesuaikan dan mendapati pelaksanaan primitif sedia ada terlalu mudah untuk keperluan mereka. Pengguna umum tidak akan menggunakan kelas asas secara langsung.

<span id="implementations"></span>
### Pelaksanaan
Berikut adalah pelaksanaan kelas asas primitif:

- Primitif Qiskit Runtime ([`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) dan [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)) menyediakan pelaksanaan yang lebih canggih (contohnya, dengan memasukkan mitigasi ralat) sebagai perkhidmatan berasaskan awan. Pelaksanaan primitif asas ini digunakan untuk mengakses perkakasan IBM Quantum&reg;. Ia diakses melalui IBM Qiskit Runtime.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) dan [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Pelaksanaan rujukan primitif yang menggunakan simulator terbina dalam Qiskit. Ia dibina dengan modul Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), menghasilkan keputusan berdasarkan simulasi statevector yang ideal. Ia diakses melalui Qiskit.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) dan [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Anda boleh menggunakan kelas ini untuk "membungkus" mana-mana sumber pengkomputeran kuantum menjadi primitif. Ini membolehkan anda menulis kod bergaya primitif untuk pembekal yang belum lagi mempunyai antara muka berasaskan primitif. Kelas-kelas ini boleh digunakan sama seperti Sampler dan Estimator biasa, kecuali ia perlu dimulakan dengan argumen `backend` tambahan untuk memilih komputer kuantum yang hendak digunakan. Ia diakses menggunakan Qiskit.

## Manfaat primitif Qiskit
Dengan primitif, pengguna Qiskit boleh menulis kod kuantum untuk QPU tertentu tanpa perlu mengurus setiap perincian secara eksplisit. Selain itu, disebabkan lapisan abstraksi tambahan, anda mungkin dapat mengakses keupayaan perkakasan lanjutan pembekal tertentu dengan lebih mudah. Contohnya, dengan primitif Qiskit Runtime, anda boleh memanfaatkan kemajuan terkini dalam mitigasi dan penindasan ralat dengan menukar pilihan seperti [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) primitif, daripada membina pelaksanaan teknik ini sendiri.

Bagi pembekal perkakasan, melaksanakan primitif secara asli bermakna anda boleh menyediakan pengguna anda dengan cara "sedia guna" yang lebih mudah untuk mengakses ciri perkakasan anda seperti teknik pasca-pemprosesan lanjutan. Oleh itu, pengguna anda lebih mudah mendapat manfaat daripada keupayaan terbaik perkakasan anda.

## Perincian primitif
Seperti yang diterangkan sebelum ini, semua primitif dicipta daripada kelas asas; oleh itu, mereka mempunyai struktur dan penggunaan am yang sama. Contohnya, format input untuk semua primitif Estimator adalah sama. Walau bagaimanapun, terdapat perbezaan dalam pelaksanaan yang menjadikan setiap satu unik.

> **Note:** Oleh sebab kebanyakan pengguna mengakses primitif Qiskit Runtime, contoh-contoh dalam selebihnya bahagian ini adalah berdasarkan primitif Qiskit Runtime.

<span id="estimator"></span>
### Estimator

Primitif Estimator mengira nilai jangkaan untuk satu atau lebih observable berkenaan keadaan yang disediakan oleh Circuit kuantum. Circuit boleh diparametrikan, selagi nilai parameter juga disediakan sebagai input kepada primitif.

Input adalah tatasusunan [PUB.](/guides/primitive-input-output#pubs) Setiap PUB adalah dalam format:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

di mana `parameter values` pilihan boleh berupa senarai atau parameter tunggal. Pelaksanaan Estimator yang berbeza menyokong pelbagai pilihan konfigurasi. Jika input mengandungi pengukuran, ia akan diabaikan.

Output adalah [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) yang mengandungi nilai jangkaan yang dikira bagi setiap pasangan, dan ralat piawai mereka, dalam bentuk `PubResult`. Setiap `PubResult` mengandungi data dan metadata.

Estimator menggabungkan elemen daripada observable dan nilai parameter dengan mengikuti peraturan penyiaran NumPy seperti yang diterangkan dalam topik [Input dan output primitif](primitive-input-output#broadcasting-rules).

Contoh:

In [1]:
# This cell is hidden from users, it creates the circuits and observables to run

from qiskit_ibm_runtime import EstimatorV2, SamplerV2, QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

service = QiskitRuntimeService()
backend = service.least_busy()
phi = Parameter("phi")

circuit1 = random_circuit(10, 5, seed=12345)
circuit1.rzz(phi, 1, 2)
observable1 = SparsePauliOp.from_sparse_list(
    [("ZXYZ", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values1 = np.random.uniform(size=5).T

circuit2 = random_circuit(10, 5, seed=12345)
circuit2.rzz(phi, 1, 2)
observable2 = SparsePauliOp.from_sparse_list(
    [("XZYX", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values2 = np.random.uniform(size=5).T

shots1 = 164
shots2 = 1024

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
circuit1 = pm.run(circuit1)
circuit2 = pm.run(circuit2)
observable1 = observable1.apply_layout(circuit1.layout)
observable2 = observable2.apply_layout(circuit2.layout)

In [2]:
estimator = EstimatorV2(mode=backend)
estimator_job = estimator.run(
    [
        (circuit1, observable1, param_values1),
        (circuit2, observable2, param_values2),
    ]
)

<span id="sampler"></span>
### Sampler

Tugas teras Sampler ialah mengambil sampel daftar output daripada pelaksanaan satu atau lebih Circuit kuantum. Circuit input boleh diparametrikan, selagi nilai parameter juga disediakan sebagai input kepada primitif.

Input adalah satu atau lebih [PUB,](/guides/primitive-input-output#pubs) dalam format:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

di mana boleh terdapat berbilang item `parameter values`, dan setiap item boleh berupa tatasusunan atau parameter tunggal, bergantung pada Circuit yang dipilih. Selain itu, input mesti mengandungi pengukuran.

Output adalah kiraan atau pengukuran per-shot, sebagai objek [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), tanpa pemberat. Walau bagaimanapun, kelas keputusan mempunyai kaedah untuk mengembalikan sampel berwajaran, seperti kiraan. Lihat [Input dan output primitif](primitive-input-output#broadcasting-rules) untuk butiran penuh.

Contoh:

In [3]:
# This cell is hidden from users, add measurement instructions to circuits
circuit1.measure_active()
circuit2.measure_active()

In [4]:
sampler = SamplerV2(mode=backend)
sampler_job = sampler.run(
    [
        (circuit1, param_values1, shots1),
        (circuit2, param_values2, shots2),
    ]
)

## Langkah seterusnya
> **Tip:** - Baca [Mulakan dengan primitif](get-started-with-primitives) untuk melaksanakan primitif dalam kerja anda.
>     - Semak [contoh primitif](primitives-examples) yang terperinci.
>     - Berlatih dengan primitif melalui [pelajaran Fungsi kos](/learning/courses/variational-algorithm-design/cost-functions) dalam IBM Quantum Learning.
>     - Lihat [rujukan API EstimatorV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) dan [rujukan API SamplerV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2).
>     - Baca [Migrasi ke primitif V2](/guides/v2-primitives).